##Ingestión del archivo "olist_customers_dataset"

In [0]:
%sql
USE e_commerce.bronze;

In [0]:
%run "../src/utils/common_functions"

In [0]:
%run "../src/utils/configuration"

In [0]:
%run "../src/utils/read"

In [0]:
%run "../src/utils/rules"

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp, input_file_name

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date","2025-10-25")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

In [0]:
customer_schema = StructType(fields=[
    StructField("customer_id", StringType(), False),
    StructField("customer_unique_id", StringType(), False),
    StructField("customer_zip_code_prefix", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])

In [0]:
path = f"{land_folder_path}/{v_file_date}/olist_customers_dataset_{v_file_date}.csv"

In [0]:
options = {
    "header": True,
    "sep": ","
}

In [0]:
df_customer = read_data(path, format="csv", schema=customer_schema, options=options)

In [0]:
reglas = []

In [0]:
if df_customer.isEmpty():

    no_registros(df_customer, "customer_id", reglas, "customer", v_file_date)

    df_reglas = spark.createDataFrame(reglas)
    df_reglas = df_reglas.withColumn("validation_date", current_timestamp())

    df_reglas.write.mode("append").format("delta").saveAsTable("bronze.log_transformation")

In [0]:
df_customer_bronze = add_ingestion_timestamp(df_customer)\
    .withColumn("environment", lit(v_environment))\
    .withColumn("file_date", lit(v_file_date))\
    .withColumn("source_name", input_file_name())

In [0]:
try:
    df_customer_bronze.write.format("delta").option("mergeSchema", "true").mode("append").partitionBy("file_date").option("path", f"{bronze_folder_path}/customer").saveAsTable("bronze.customer")
except Exception as e:
    error_msg = handle_error(e, dbutils)
    print(error_msg)
    raise

In [0]:
%sql
SELECT file_date, COUNT(1) FROM bronze.customer GROUP BY file_date ORDER BY 1;